# Concurrency Utilities Demo (`parallel_utilities`)

El módulo `concurrency.parallel_utilities` (`ds_utils.concurrency.parallel_utilities`) proporciona utilidades para la ejecución multihilo y concurrente de funciones sobre colecciones iterables mediante `ThreadPoolExecutor`.

### Características principales:
- **`parallelize`**: Distribución concurrente de tareas iterables en un grupo de hilos (`ThreadPoolExecutor`).
- **Separación limpia de resultados y errores**: Devuelve una tupla `(results, errors)` que evita que una excepción individual cancele o interrumpa el procesamiento del resto del lote.
- **Estructura tipada de salidas**: Utiliza `TypedDict` (`ParallelResult` y `ParallelError`) con genéricos `[T, R]` para que las respuestas y errores tengan un esquema predecible y validable.
- **`CPU_THREADS`**: Constante que detecta automáticamente los hilos de CPU disponibles en el sistema (`os.cpu_count() or 1`) para configurar el nivel de paralelismo predeterminado.

### Contenido del Cuaderno:
1. **Configuración e Importaciones**: Preparación del entorno y verificación de hilos disponibles.
2. **Inspección de Constantes y Tipos**: Exploración de `CPU_THREADS`, `ParallelResult` y `ParallelError`.
3. **Procesamiento Concurrente Básico**: Demostración de aceleración multihilo en tareas limitadas por I/O.
4. **Manejo Robusto de Excepciones**: Captura de errores heterogéneos sin detener la ejecución global.
5. **Control de Concurrencia con `max_workers`**: Comparativa de tiempos entre ejecución secuencial y concurrente.
6. **Caso de Uso en Data Science**: Validación y cálculo de estadísticas descriptivas sobre lotes de datos.

## 1. Configuración e Importaciones

Cargamos las extensiones de recarga automática y las utilidades del módulo `parallel_utilities`.

In [ ]:
# RECARGA AUTOMÁTICA DE MÓDULOS EXTERNOS
%load_ext autoreload
%autoreload 2

import time
import pandas as pd

from ds_utils.core.parallel_utilities import (
    CPU_THREADS,
    ParallelError,
    ParallelResult,
    parallelize,
)

print(f"Hilos de CPU disponibles en el entorno (CPU_THREADS): {CPU_THREADS}")

Hilos de CPU disponibles en el entorno (CPU_THREADS): 12


## 2. Inspección de Constantes y Tipos de Retorno

El módulo define estructuras tipadas mediante `TypedDict` que formalizan la salida de cada elemento:
- `ParallelResult`: almacena `item` (la entrada original de tipo `T`) y `result` (la salida calculada de tipo `R`).
- `ParallelError`: almacena `item` (la entrada original de tipo `T`) y `error` (un diccionario con `type` y `message` de la excepción capturada).

In [2]:
# Inspección de la estructura de tipos del módulo
print("Esquema de tipos:")
print(f"- ParallelResult annotations: {ParallelResult.__annotations__}")
print(f"- ParallelError annotations:  {ParallelError.__annotations__}")

Esquema de tipos:
- ParallelResult annotations: {'item': T, 'result': R}
- ParallelError annotations:  {'item': T, 'error': dict[str, str]}


## 3. Procesamiento Concurrente Básico (Simulación de I/O)

Simulamos una función de consulta o telemetría que requiere un tiempo de espera de red (0.05 s por elemento).
Con 8 elementos, una ejecución puramente secuencial tomaría ~0.40 segundos, mientras que la ejecución concurrente con hilos se completa en una fracción de ese tiempo.

In [3]:
def fetch_sensor_telemetry(sensor_id: str) -> dict:
    """Simula una llamada I/O (lectura de red, base de datos o API)."""
    time.sleep(0.05)
    return {
        "sensor_id": sensor_id,
        "status": "online",
        "temperature": 21.5,
        "humidity": 45.0,
    }

sensor_list = [f"sensor_{i:02d}" for i in range(1, 9)]

start_time = time.perf_counter()
results, errors = parallelize(
    func=fetch_sensor_telemetry,
    iterable=sensor_list,
    max_workers=4,
)
elapsed_time = time.perf_counter() - start_time

print(f"Total elementos procesados exitosamente: {len(results)}")
print(f"Total errores: {len(errors)}")
print(f"Tiempo de ejecución concurrente: {elapsed_time:.3f} s (frente a ~{0.05 * len(sensor_list):.2f} s secuencial)")

# Visualizamos los resultados obtenidos en un DataFrame
df_telemetry = pd.DataFrame([r["result"] for r in results])
display(df_telemetry)

Total elementos procesados exitosamente: 8
Total errores: 0
Tiempo de ejecución concurrente: 0.103 s (frente a ~0.40 s secuencial)


,sensor_id,status,temperature,humidity
0,sensor_01,online,21.5,45.0
1,sensor_03,online,21.5,45.0
2,sensor_02,online,21.5,45.0
3,sensor_04,online,21.5,45.0
4,sensor_05,online,21.5,45.0
5,sensor_06,online,21.5,45.0
6,sensor_07,online,21.5,45.0
7,sensor_08,online,21.5,45.0


## 4. Manejo Robusto de Excepciones y Separación de Errores

Una ventaja fundamental de `parallelize` es que **aísla las excepciones**:
- Si un elemento lanza un error (`ZeroDivisionError`, `ValueError`, etc.), los demás elementos siguen ejecutándose con normalidad.
- Los elementos exitosos se acumulan en la lista `results`.
- Los elementos fallidos se registran en `errors` preservando el ítem original y el detalle del error (`type` y `message`).

In [4]:
def compute_efficiency_ratio(record: dict) -> float:
    """Calcula el ratio de eficiencia energética. Lanza excepciones ante datos anómalos."""
    device = record["device"]
    output = record["output"]
    input_power = record["input_power"]

    if input_power == 0:
        raise ZeroDivisionError(f"Potencia de entrada igual a cero en '{device}'")
    if output < 0:
        raise ValueError(f"Salida de energía negativa inválida ({output}) en '{device}'")

    return round(output / input_power, 4)

records_data = [
    {"device": "Turbina_1", "output": 150.0, "input_power": 200.0},
    {"device": "Turbina_2", "output": 0.0,   "input_power": 0.0},       # Lanza ZeroDivisionError
    {"device": "Turbina_3", "output": 180.0, "input_power": 210.0},
    {"device": "Turbina_4", "output": -25.0, "input_power": 100.0},     # Lanza ValueError
    {"device": "Turbina_5", "output": 95.0,  "input_power": 120.0},
]

results, errors = parallelize(
    func=compute_efficiency_ratio,
    iterable=records_data,
    max_workers=3,
)

print(f"Tareas completadas con éxito: {len(results)}")
print(f"Tareas con excepción capturada: {len(errors)}")

Tareas completadas con éxito: 3
Tareas con excepción capturada: 2


### Inspección de resultados exitosos vs. errores capturados

Podemos estructurar ambos listados directamente en DataFrames para auditoría y visualización clara.

In [5]:
# Tabla de ejecuciones exitosas
df_success = pd.DataFrame([
    {"device": r["item"]["device"], "efficiency_ratio": r["result"]}
    for r in results
]).sort_values("device").reset_index(drop=True)

print("=== TAREAS EXITOSAS ===")
display(df_success)

# Tabla de auditoría de errores
df_errors = pd.DataFrame([
    {
        "device": e["item"]["device"],
        "error_type": e["error"]["type"],
        "error_message": e["error"]["message"],
    }
    for e in errors
]).sort_values("device").reset_index(drop=True)

print("=== AUDITORÍA DE ERRORES CAPTURADOS ===")
display(df_errors)

=== TAREAS EXITOSAS ===


,device,efficiency_ratio
0,Turbina_1,0.7500
1,Turbina_3,0.8571
2,Turbina_5,0.7917


=== AUDITORÍA DE ERRORES CAPTURADOS ===


,device,error_type,error_message
0,Turbina_2,ZeroDivisionError,Potencia de entrada igual a cero en 'Turbina_2'
1,Turbina_4,ValueError,Salida de energía negativa inválida (-25.0) en...


## 5. Control de Concurrencia con `max_workers`

El argumento `max_workers` permite ajustar el grado de paralelismo según los requisitos:
- Limitar `max_workers` (por ejemplo, a 2 o 4) resulta útil para evitar sobrecargar servicios externos con cuotas de petición (*rate limit*) o servidores de base de datos.
- Utilizar `max_workers=CPU_THREADS` aprovecha al máximo los recursos multihilo del sistema.
- A continuación comparamos la ejecución con `max_workers=1` (equivalente a ejecución secuencial) frente a `max_workers=4`.

In [6]:
def simulate_light_io(task_id: int) -> int:
    time.sleep(0.02)
    return task_id * 2

tasks = list(range(12))

# Ejecución con 1 worker (secuencial)
t0 = time.perf_counter()
res_1, _ = parallelize(simulate_light_io, tasks, max_workers=1)
time_1 = time.perf_counter() - t0

# Ejecución con 4 workers (concurrente)
t0 = time.perf_counter()
res_4, _ = parallelize(simulate_light_io, tasks, max_workers=4)
time_4 = time.perf_counter() - t0

print(f"Tiempo con max_workers=1: {time_1:.3f} s")
print(f"Tiempo con max_workers=4: {time_4:.3f} s")
print(f"Factor de aceleración:    {time_1 / time_4:.2f}x")

Tiempo con max_workers=1: 0.244 s
Tiempo con max_workers=4: 0.064 s
Factor de aceleración:    3.83x


## 6. Caso de Uso Completo: Procesamiento de Lotes en Data Science

En flujos de ciencia de datos es común recibir colecciones de lotes o particiones de series temporales. A continuación procesamos varios lotes calculando métricas estadísticas (`mean`, `std`, `min`, `max`), gestionando lotes vacíos o defectuosos de forma automática.

In [7]:
def process_batch(batch: dict) -> dict:
    """Valida y resume métricas sobre una partición de lecturas."""
    batch_id = batch["batch_id"]
    values = batch["readings"]

    if not values:
        raise ValueError(f"El lote '{batch_id}' no contiene observaciones válidas")

    s = pd.Series(values)
    return {
        "batch_id": batch_id,
        "num_observaciones": len(s),
        "media": round(float(s.mean()), 2),
        "desv_estandar": round(float(s.std()), 2) if len(s) > 1 else 0.0,
        "minimo": round(float(s.min()), 2),
        "maximo": round(float(s.max()), 2),
    }

dataset_batches = [
    {"batch_id": "Lote_Norte", "readings": [21.1, 21.5, 22.0, 21.8, 21.9]},
    {"batch_id": "Lote_Sur",   "readings": [19.5, 19.8, 20.1, 19.9]},
    {"batch_id": "Lote_Este",  "readings": []},  # Lote vacío: genera excepción controlada
    {"batch_id": "Lote_Oeste", "readings": [25.0, 25.2, 24.9, 25.1, 25.3, 25.0]},
]

results, errors = parallelize(process_batch, dataset_batches)

# Consolidación en DataFrame de métricas
df_summary = pd.DataFrame([r["result"] for r in results]).sort_values("batch_id").reset_index(drop=True)
print("Resumen estadístico consolidado por lote:")
display(df_summary)

if errors:
    print("\nLotes descartados por validación fallida:")
    for err in errors:
        print(f"- {err['item']['batch_id']}: [{err['error']['type']}] {err['error']['message']}")

Resumen estadístico consolidado por lote:


,batch_id,num_observaciones,media,desv_estandar,minimo,maximo
0,Lote_Norte,5,21.66,0.36,21.1,22.0
1,Lote_Oeste,6,25.08,0.15,24.9,25.3
2,Lote_Sur,4,19.82,0.25,19.5,20.1



Lotes descartados por validación fallida:
- Lote_Este: [ValueError] El lote 'Lote_Este' no contiene observaciones válidas
